# AI-Powered Financial Fraud Detection & Risk Analytics System

## Notebook 1: Data Understanding & Exploratory Data Analysis (EDA)

### Objective
The purpose of this notebook is to understand the PaySim financial transaction dataset, analyze fraud patterns, identify business insights, and prepare the foundation for feature engineering and machine learning modeling.

Dataset: PaySim Mobile Money Fraud Dataset

Total Records: 6,362,620

Target Variable: isFraud

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

plt.style.use("ggplot")

print("Libraries Loaded Successfully")

In [ ]:
df = pd.read_csv("../data/raw/paysim.csv")
df.head()

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

In [ ]:
df.describe()
df.describe(include="object")

In [ ]:
missing = df.isnull().sum()
missing[missing > 0]

if missing[missing > 0].size == 0:
    print("No Missing Values Found")

In [ ]:
duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

## Fraud Distribution Analysis

In [ ]:
fraud_counts = df["isFraud"].value_counts()

fraud_percentage = (
    df["isFraud"].value_counts(normalize=True)
    * 100
)

print(fraud_percentage)

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    x="isFraud",
    data=df
)

plt.savefig("../reports/figures/fraud_vs_nonfraud.png", dpi=300, bbox_inches="tight")
plt.title("Fraud vs Non-Fraud Transactions")
plt.show()

## Transaction Type Analysis

In [ ]:
df["type"].value_counts()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    y="type",
    data=df,
    order=df["type"].value_counts().index
)

plt.savefig("../reports/figures/transaction_type_distribution.png", dpi=300, bbox_inches="tight")
plt.title("Transaction Type Distribution")
plt.show()

In [ ]:
# Fraud by Transaction Type

fraud_type = pd.crosstab(
    df["type"],
    df["isFraud"]
)

fraud_type

In [ ]:
fraud_percentage_type = (
    pd.crosstab(
        df["type"],
        df["isFraud"],
        normalize="index"
    ) * 100
)

fraud_percentage_type

### Observation

Fraud is concentrated mainly in:

- TRANSFER transactions
- CASH_OUT transactions

No fraud transactions are observed in:

- PAYMENT
- CASH_IN
- DEBIT

This indicates that fraudsters primarily move money through transfer and cash withdrawal operations.

In [ ]:
# Transaction Amount Analysis

df.groupby("isFraud")["amount"].describe()

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(
    x="isFraud",
    y="amount",
    data=df
)

plt.yscale("log")
plt.title("Transaction Amount Distribution")
plt.savefig("../reports/figures/transaction_amount_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Correlation Analysis

numeric_df = df.select_dtypes(
    include=np.number
)

corr_matrix = numeric_df.corr()

corr_matrix

In [ ]:
plt.figure(figsize=(12,8))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm"
)

plt.title(
    "Correlation Matrix"
)
plt.savefig("../reports/figures/correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Fraud Correlation Analysis

corr_matrix["isFraud"]\
.sort_values(
    ascending=False
)

# Key Business Insights

### 1. Fraud Rate

Only 0.129% of all transactions are fraudulent.

This indicates a highly imbalanced dataset.

---

### 2. High Risk Transaction Types

TRANSFER and CASH_OUT transactions contribute to nearly all fraud cases.

---

### 3. Fraud Amounts

Fraudulent transactions generally involve larger amounts compared to normal transactions.

---

### 4. Class Imbalance

The dataset contains extremely few fraud transactions compared to genuine transactions.

Advanced balancing techniques will be required before model training.

---

### 5. Feature Engineering Opportunity

Current variables alone may not be sufficient.

Derived features such as:

- Balance Error
- Transaction Velocity
- Hour of Transaction
- Risk Score

can significantly improve fraud detection performance.

In [ ]:
eda_summary = {
    "total_rows": len(df),
    "total_columns": len(df.columns),
    "fraud_cases": df["isFraud"].sum(),
    "fraud_percentage":
        round(
            df["isFraud"].mean()*100,
            4
        )
}

pd.DataFrame(
    [eda_summary]
).to_csv(
    "../reports/eda/eda_summary.csv",
    index=False
)

print("EDA Summary Saved")